# Sentinel-2 optical — true colour & NDVI

Reads a **monthly-median** Sentinel-2 cube (Zarr) and makes two quick figures: a true-colour composite and NDVI (map + time series).

This notebook uses the **bundled demo pack** (`hackathon_demo_data/`), so it runs from local disk with **no S3 and no credentials**. The Sentinel-2 example is a small clip of the **A_koranga_forks_nz** (New Zealand) AOI — note this differs from the other notebooks, which use the D_Chablis_Vineyard AOI. See the repo README for how to download and unzip the pack.

The demo cube `sentinel2/cube.zarr`:
- Bands `B02`-`B12` (uint16 surface reflectance, scale 1/10000) plus `n_obs`.
- A **200 x 200** pixel window, **24 monthly steps (2021-2022)**, 10 m, SCL-cloud-masked before the median.
- `n_obs == 0` means *no clear observation that month* - treat those pixels as nodata.

> Note: koranga is in the southern hemisphere, so its NDVI seasonal cycle is phase-shifted relative to a European site — that is expected.

> ⚠️ **Read before any multi-year analysis:** reflectance is **not baseline-harmonised**, so
> there is an artificial step at **2022-01-25** (Processing Baseline 04.00). The demo window
> deliberately straddles that date. See the "Important" section below the NDVI time series for
> the cause and the one-line fix.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# This notebook reads the bundled demo pack from local disk - no S3, no credentials.
# Download hackathon_demo_data.zip (see the repo README) and unzip it next to this notebook so
# that ./hackathon_demo_data/ exists.
DATA_DIR = Path("hackathon_demo_data")

In [ ]:
AOI = "A_koranga_forks_nz"  # the bundled S2 demo AOI (New Zealand)
ds = xr.open_zarr(str(DATA_DIR / "sentinel2" / "cube.zarr"))
print("dims:", dict(ds.sizes))
print("bands:", [v for v in ds.data_vars if v != "n_obs"])
print("time:", str(ds.time.values[0])[:10], "->", str(ds.time.values[-1])[:10])
ds

## True-colour composite

Pick the month with the most clear observations, stack R=B04, G=B03, B=B02, and scale reflectance to 0-1 for display. Pixels with `n_obs == 0` are masked to white.

In [ ]:
# month with the most clear observations, restricted to PRE-2022.
# This cube is not baseline-harmonised (see the warning below the time series), so
# DN/10000 is only valid before the baseline-04.00 cutover. We pick a pre-2022 month
# here so the true-colour stretch and the NDVI map below are radiometrically correct.
obs = ds["n_obs"].median(dim=("y", "x")).where(ds.time < np.datetime64("2022-01-25"))
t = int(obs.argmax("time"))
scene = ds.isel(time=t)
month = str(scene.time.values)[:7]
print("selected month:", month)

rgb = np.stack([scene["B04"], scene["B03"], scene["B02"]], axis=-1).astype("float32")
rgb = np.clip(rgb / 3000.0, 0, 1)          # 3000 ~ a good reflectance stretch
rgb[scene["n_obs"].values == 0] = 1.0       # nodata -> white

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb)
ax.set_title(f"Sentinel-2 true colour - {AOI} - {month}")
ax.axis("off")
fig.savefig("01_s2_true_colour.png", dpi=120, bbox_inches="tight")

## NDVI map

`NDVI = (B08 - B04) / (B08 + B04)`, ranging -1 to 1. High NDVI (green) = dense vegetation.

In [ ]:
b8 = scene["B08"].astype("float32")
b4 = scene["B04"].astype("float32")
ndvi = (b8 - b4) / (b8 + b4)
ndvi = ndvi.where(scene["n_obs"] > 0)        # drop nodata

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax.set_title(f"NDVI - {AOI} - {month}")
ax.axis("off")
fig.colorbar(im, ax=ax, shrink=0.7, label="NDVI")
fig.savefig("01_s2_ndvi.png", dpi=120, bbox_inches="tight")

## NDVI time series

Mean NDVI over a central window across all 24 months of the demo cube. (On the full hackathon cubes — 108 months over the whole AOI — you would window or chunk to keep reads light; here the clip is already tiny.)

In [ ]:
ny, nx = ds.sizes["y"], ds.sizes["x"]
win = ds.isel(y=slice(ny // 2 - 100, ny // 2 + 100), x=slice(nx // 2 - 100, nx // 2 + 100))

b8 = win["B08"].astype("float32")
b4 = win["B04"].astype("float32")
ndvi = ((b8 - b4) / (b8 + b4)).where(win["n_obs"] > 0)
series = ndvi.mean(dim=("y", "x")).compute()   # streams just the window

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(series.time, series, marker="o", ms=3, lw=1)
ax.set_title(f"Mean NDVI (central window) - {AOI}")
ax.set_ylabel("NDVI")
ax.grid(alpha=0.3)
fig.savefig("01_s2_ndvi_series.png", dpi=120, bbox_inches="tight")

## ⚠️ Important: the step at the start of 2022 is a processing artefact

If your NDVI series spans 2022 you will see a sharp **downward step around Jan/Feb 2022**.
This is **not** a real land-cover change — it is the **Sentinel-2 Processing Baseline 04.00**
change (effective **2022-01-25**), which added an offset of `BOA_ADD_OFFSET = -1000` to L2A
reflectance.

These cubes were built from Microsoft Planetary Computer via stackstac, and **MPC's
`sentinel-2-l2a` assets carry no `raster:bands` scale/offset metadata**, so that offset is
**never applied**. Scenes from 2022-01-25 onwards therefore sit ~**+1000 DN** higher than
earlier scenes. Both bands shift up by ~1000, but NDVI's denominator grows while its
numerator does not, so vegetation NDVI is pushed *down*.

**Always harmonise before comparing across 2022-01-25.** Subtract 1000 (clamp at 0) from
the optical bands of scenes on/after that date. (See `tf-data-ml-utils`
`science_ai_sdk/README.md` → "Known caveat".)

In [ ]:
# Harmonise: subtract the baseline-04.00 offset from scenes on/after 2022-01-25.
OPTICAL = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
BASELINE_CUTOVER = np.datetime64("2022-01-25")


def harmonise_s2(cube):
    """Undo the Processing Baseline 04.00 +1000 DN offset (clamp at 0)."""
    post = cube["time"] >= BASELINE_CUTOVER
    out = cube.copy()
    for b in OPTICAL:
        if b in out:
            out[b] = xr.where(post, (cube[b].astype("float32") - 1000).clip(min=0), cube[b])
    return out


win_h = harmonise_s2(win)
b8h = win_h["B08"].astype("float32")
b4h = win_h["B04"].astype("float32")
series_h = (((b8h - b4h) / (b8h + b4h)).where(win["n_obs"] > 0)
            .mean(dim=("y", "x")).compute())

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(series.time, series, marker="o", ms=3, lw=1, color="0.7", label="raw (as stored)")
ax.plot(series_h.time, series_h, marker="o", ms=3, lw=1, color="C2", label="baseline-harmonised")
ax.axvline(BASELINE_CUTOVER, color="C3", ls="--", lw=1, label="baseline 04.00 (2022-01-25)")
ax.set_title(f"Mean NDVI - raw vs harmonised - {AOI}")
ax.set_ylabel("NDVI")
ax.legend(loc="lower left", fontsize=8)
ax.grid(alpha=0.3)
fig.savefig("01_s2_ndvi_series_harmonised.png", dpi=120, bbox_inches="tight")